# Sample Notebook: OLMoEarth Pixel Embedding Generation

This notebook shows how to use the Pixelverse model registry to load OLMoEarth and generate **pixel embeddings** from Sentinel-2 time-series imagery.

Current wrapper contract:
- image input: `B, T, C, H, W`
- timestamps input: `B, T, 3` as `[day, month, year]` (month is zero-indexed)
- output: `B, H, W, D` (one embedding per input pixel by default, `patch_size=1`)


In [ ]:
import numpy as np
import torch
import xarray as xr

import pixelverse as pv

In [ ]:
MODEL_NAME = "olmoearth_nano"
model, transforms = pv.create_model(MODEL_NAME)
weights = pv.get_weights(MODEL_NAME)
model.eval()
weights.meta

## Required Sentinel-2 Band Order

OLMoEarth expects 12 Sentinel-2 bands in this channel order (`C`):

`B02, B03, B04, B08, B05, B06, B07, B8A, B11, B12, B01, B09`

The helper below accepts either:
- Pixelverse `get_s2_time_series(...)` datasets (common STAC names like `blue`, `nir09`)
- datasets already renamed to OLMoEarth band IDs (`B02`, `B03`, ...)


In [ ]:
OLMOEARTH_BANDS = [
    "B02",
    "B03",
    "B04",
    "B08",
    "B05",
    "B06",
    "B07",
    "B8A",
    "B11",
    "B12",
    "B01",
    "B09",
]

S2_COMMON_NAME_TO_OLMOEARTH = {
    "coastal": "B01",
    "blue": "B02",
    "green": "B03",
    "red": "B04",
    "nir": "B08",
    "rededge1": "B05",
    "rededge2": "B06",
    "rededge3": "B07",
    "nir08": "B8A",
    "swir16": "B11",
    "swir22": "B12",
    "nir09": "B09",
}


def build_olmoearth_inputs_from_xarray(ds: xr.Dataset) -> tuple[torch.Tensor, torch.Tensor]:
    if set(S2_COMMON_NAME_TO_OLMOEARTH).issubset(ds.data_vars):
        ds = ds.rename(S2_COMMON_NAME_TO_OLMOEARTH)

    missing = [band for band in OLMOEARTH_BANDS if band not in ds]
    if missing:
        raise KeyError(f"Dataset is missing required bands: {missing}")

    arr = ds[OLMOEARTH_BANDS].to_array(dim="band").transpose("time", "band", "y", "x").values
    x = torch.from_numpy(arr).unsqueeze(0).float()  # (B, T, C, H, W)

    t = ds.time.dt
    timestamps_np = np.stack(
        [
            t.day.values.astype(np.int64),
            (t.month.values - 1).astype(np.int64),
            t.year.values.astype(np.int64),
        ],
        axis=-1,
    )
    timestamps = torch.from_numpy(timestamps_np).unsqueeze(0)
    return x, timestamps

## Minimal Synthetic Example (Runnable Everywhere)

This verifies normalization + wrapper inference and shows the default pixel-embedding output shape.


In [ ]:
B, T, C, H, W = 1, 4, 12, 16, 16
x = torch.randint(0, 10000, (B, T, C, H, W), dtype=torch.int32).float()
timestamps = torch.tensor(
    [[[15, 0, 2024], [15, 1, 2024], [15, 2, 2024], [15, 3, 2024]]],
    dtype=torch.long,
)

with torch.no_grad():
    embeddings = model(transforms(x), timestamps=timestamps)

embeddings.shape  # (B, H, W, D) with patch_size=1

## Convert To Xarray (Feature-First)

For storage/visualization, it is often convenient to reorder the output to `(feature, y, x)`.


In [ ]:
embeddings_da = xr.DataArray(
    embeddings[0].cpu().numpy(),
    dims=["y", "x", "feature"],
).transpose("feature", "y", "x")
embeddings_da

## Example With `get_s2_time_series`

Pass model-specific STAC band names from weights metadata so the download matches the registry model requirements.


In [ ]:
# Example (not executed here):
# from pixelverse.download.sentinel2 import get_s2_time_series
#
# weights = pv.get_weights("olmoearth_nano")
# ds = get_s2_time_series(
#     bbox=...,
#     year=2024,
#     bands=weights.meta["s2_stac_bands"],
# )
# x, timestamps = build_olmoearth_inputs_from_xarray(ds)
# with torch.no_grad():
#     embeddings = model(transforms(x), timestamps=timestamps)


## Save Embeddings

The wrapper returns one embedding grid per input chip (`B, H, W, D`). Save each grid as NumPy or convert to xarray for raster workflows.


In [ ]:
np.save("olmoearth_pixel_embeddings.npy", embeddings.cpu().numpy())